8.2 Data Quality Checks

Goal:
    . Load a real world dataset
    . Run common data quality checks
    . Build a simple data quality checklist

1. Setup

In [32]:
%pip install -q pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

**Load the dataset**


**Titanic.csv**

In [34]:
df= pd.read_csv("titanic_dataset.csv")

In [35]:
df.shape

(891, 15)

In [36]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


**3. Adding few columns for demonstarting data issues**

In [37]:
# Adding a new column passenger id
df["passenger_id"] = np.arange(1, len(df)+1)

#

In [38]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,passenger_id
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,1
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,3
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,4
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,5


In [39]:
# create a dirty verion of embark_town
df["embark_town_dirty"] = df["embark_town"].copy()

In [40]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,passenger_id,embark_town_dirty
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,1,Southampton
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2,Cherbourg
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,3,Southampton
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,4,Southampton
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,5,Southampton


In [41]:
#select few random rows(only non-missing)
sample_idx = df["embark_town_dirty"].dropna().sample(6, random_state=42).index

In [42]:
sample_idx

Index([281, 435, 39, 418, 585, 804], dtype='int64')

In [48]:
# apply simple dirty transformations
df.loc[sample_idx[0:2], "embark_town_dirty"] = df.loc[sample_idx[0:2], "embark_town_dirty"].str.upper()
df.loc[sample_idx[2:4], "embark_town_dirty"] = df.loc[sample_idx[2:4], "embark_town_dirty"].str.strip().apply(lambda x: f" {x} ")
df.loc[sample_idx[4:6], "embark_town_dirty"] = "unknown"

In [50]:
df.iloc[281]["embark_town_dirty"]

'SOUTHAMPTON'

**4. Data quality checklist**
1. Basic dataset overview
2. Missing values summary
3. Duplicates
4. Data type validation
5. Constant and quasi constant columns
6. ID like columns
7. String inconsistencies
8. High null columns


**4.1 Basic dataset overview** 

In [52]:
df.shape

(891, 17)

In [53]:
df.dtypes

survived               int64
pclass                 int64
sex                   object
age                  float64
sibsp                  int64
parch                  int64
fare                 float64
embarked              object
class                 object
who                   object
adult_male              bool
deck                  object
embark_town           object
alive                 object
alone                   bool
passenger_id           int64
embark_town_dirty     object
dtype: object

In [54]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone', 'passenger_id', 'embark_town_dirty'],
      dtype='object')

In [55]:
df.nunique()

survived               2
pclass                 3
sex                    2
age                   88
sibsp                  7
parch                  7
fare                 248
embarked               3
class                  3
who                    3
adult_male             2
deck                   7
embark_town            3
alive                  2
alone                  2
passenger_id         891
embark_town_dirty      7
dtype: int64

In [56]:
df.describe()

,survived,pclass,age,sibsp,parch,fare,passenger_id
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208,446.000000
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429,257.353842
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400,223.500000
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200,446.000000
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000,668.500000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200,891.000000


**4.2 Missing value summary**

In [57]:
missing_count = df.isna().sum().sort_values(ascending= False)
missing_percent= (df.isna().mean()*100).sort_values(ascending= False)

missing_summary = pd.DataFrame(
    {
        "missing_count": missing_count,
        "missing_percent": missing_percent
    }
)

print("Missing Values Summary")
missing_summary

Missing Values Summary


,missing_count,missing_percent
deck,688,77.216611
age,177,19.865320
embark_town_dirty,2,0.224467
embark_town,2,0.224467
embarked,2,0.224467
survived,0,0.000000
sibsp,0,0.000000
sex,0,0.000000
pclass,0,0.000000
class,0,0.000000


**4.3. Checking for duplicates**

In [60]:
duplicates_mask = df.duplicated()
num_duplicates =duplicates_mask.sum()

print("Number of duplicate rows:", num_duplicates)


# if you want to remove duplicates:
# df_no_duplicates = df.drop_duplicates()

Number of duplicate rows: 0


**4.4. Data type validation**

In [62]:
expected_types={
    "survived":"int64",
    "pclass":"int64",
    "sex":"category",
    "age":"float64",
    "fare":"float64",
    "embark_town":"category",
    "passenger_id":"int64"
}

print("Data Type Validation:")
for col, expected in expected_types.items():
    if col in df.columns:
        actual=df[col].dtype
        print(f"{col}: actual={actual}, expected={expected}")

Data Type Validation:
survived: actual=int64, expected=int64
pclass: actual=int64, expected=int64
sex: actual=object, expected=category
age: actual=float64, expected=float64
fare: actual=float64, expected=float64
embark_town: actual=object, expected=category
passenger_id: actual=int64, expected=int64


In [ ]:
# if needed to change data type

#df["pclass"] = df.["pclass"].astype("int64")

**4.5. Constant and Quasi-constant column**

In [65]:
# identify column that have a constant value through out column
n_rows = len(df)
nunique = df.nunique()

constant_cols= nunique[nunique==1].index.to_list()
print("Constant columns:", constant_cols)

Constant columns: []


In [66]:
df.nunique()

survived               2
pclass                 3
sex                    2
age                   88
sibsp                  7
parch                  7
fare                 248
embarked               3
class                  3
who                    3
adult_male             2
deck                   7
embark_town            3
alive                  2
alone                  2
passenger_id         891
embark_town_dirty      7
dtype: int64

In [67]:
#quasi constant columns(not all columns are unique, 95% of column values are unique)

quasi_constant_cols = []

for col in df.columns:
    top_freq = df[col].value_counts(normalize=True, dropna=False).values[0]
    if top_freq > 0.95 and col not in constant_cols:
        quasi_constant_cols.append(col)

print("Quasi constant column(top value more than 95 percent):", quasi_constant_cols)

Quasi constant column(top value more than 95 percent): []


**4.6. ID like columns**


**Columns where number of unique values is closer to number of df rows**

In [69]:
n_rows=len(df)

id_like_col=[]

for col in df.columns:
    if df[col].nunique(dropna=False) ==n_rows:
        id_like_col.append(col)

print("ID like colums:", id_like_col)

ID like colums: ['passenger_id']


**4.7. String Incosistencies**


In [70]:
object_cols = df.select_dtypes(include=["object", "category"]).columns.to_list()
print("Object or category based columns", object_cols)

# simple clean: strip spaces convert it to lower case
df["embark_town_clean"] = (
    df["embark_town_dirty"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("unknown", np.nan)
)

Object or category based columns ['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive', 'embark_town_dirty']


In [71]:
df["embark_town_clean"]

0      southampton
1        cherbourg
2      southampton
3      southampton
4      southampton
          ...     
886    southampton
887    southampton
888    southampton
889      cherbourg
890     queenstown
Name: embark_town_clean, Length: 891, dtype: object

**4.8. High null columns**

In [72]:
high_null_threshold = 0.4 #40% or more values are missing in a column

high_null_cols= missing_summary[missing_summary["missing_percent"]>=high_null_threshold * 100]

print("Columns with high missing percentage:")
print(high_null_cols)

Columns with high missing percentage:
      missing_count  missing_percent
deck            688        77.216611


**Data quality check summary**

Data quality checklist summary
- Shape and columns checked
- Missing values summary created
- Duplicates counted and removable copy created
- Data types compared with expectations
- Constant and quasi constant columns flagged
- ID like columns detected
- String inconsistencies checked and cleaned example
- High null columns identified
- High zero numeric columns identified
